# 03 — Feature Validation

**Goal:** Verify that the features we're building actually have predictive
power for the target (Utah healthcare employment).

**Analysis:**
1. Correlation matrix of all features vs. target
2. Lag analysis — at what lag do JOLTS indicators best predict UTEDUH changes?
3. Granger causality tests for key feature→target relationships
4. Feature importance ranking

**Key question:** Do national JOLTS signals actually lead Utah employment
outcomes, or is the theoretical lead-lag relationship too noisy to exploit?

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'backend'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.ingestion import generate_dummy_series
from features.preprocessing import preprocess_all
from features.engineering import build_lag_features, compute_derived_features

In [ ]:
# Load and preprocess
raw = generate_dummy_series()

# Convert dummy DataFrames to Series indexed by date
raw_series = {}
for sid, df in raw.items():
    raw_series[sid] = df.set_index('date')['value']

processed = preprocess_all(raw_series)
print(f'Processed shape: {processed.shape}')
processed.head()

In [ ]:
# Correlation matrix
corr = processed.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix (differenced series)')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-correlation: JOLTS quit rate vs UTEDUH at various lags
if 'JTS6200QUR' in processed.columns and 'UTEDUH' in processed.columns:
    max_lag = 12
    cross_corr = []
    for lag in range(0, max_lag + 1):
        shifted = processed['JTS6200QUR'].shift(lag)
        cc = shifted.corr(processed['UTEDUH'])
        cross_corr.append({'lag': lag, 'correlation': cc})
    
    cc_df = pd.DataFrame(cross_corr)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(cc_df['lag'], cc_df['correlation'], color='steelblue')
    ax.set_xlabel('Lag (months)')
    ax.set_ylabel('Correlation')
    ax.set_title('Cross-correlation: Quit Rate (lagged) → UTEDUH')
    ax.axhline(0, color='gray', linewidth=0.5)
    plt.tight_layout()
    plt.show()
    
    best = cc_df.loc[cc_df['correlation'].abs().idxmax()]
    print(f'Best lag: {int(best["lag"])} months, correlation: {best["correlation"]:.3f}')

In [ ]:
# TODO: Granger causality tests
# from statsmodels.tsa.stattools import grangercausalitytests
# For each feature → target pair, test if the feature Granger-causes
# changes in the target at lags 1-6.

print('Granger causality tests: implement with real data')

## Findings

- [ ] Which features have the strongest correlation with UTEDUH?
- [ ] At what lag do JOLTS indicators best predict employment changes?
- [ ] Any features to drop (near-zero predictive power)?
- [ ] Proceed to `04_model_comparison.ipynb`